In [ ]:
from pathlib import Path
import pandas as pd

csv_path = Path('data/raw/ld50-smiles-descriptors-dataset.csv')
if not csv_path.is_file():
    csv_path = Path('../data/raw/ld50-smiles-descriptors-dataset.csv')

df = pd.read_csv(csv_path)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

## Clean and prepare the data

Work on a copy of `df`. `LD50` is already stored as −log10(LD50 mol/kg), so it is not transformed. Only missing target values and exact duplicate rows are removed.

### 1. Create a working copy

In [ ]:
clean_df = df.copy()
initial_rows = len(clean_df)
main_columns = ['LD50', 'NumHAcceptors']

### 2. Check missing target values

In [ ]:
missing_values = clean_df[main_columns].isnull().sum()
missing_values

### 3. Remove rows missing LD50 or NumHAcceptors

In [ ]:
clean_df = clean_df.dropna(subset=['LD50', 'NumHAcceptors'])
missing_rows_removed = initial_rows - len(clean_df)
print(f'Rows removed for missing target values: {missing_rows_removed}')
assert not clean_df[main_columns].isnull().any().any()
clean_df[main_columns].isnull().sum()

### 4. Check exact duplicate rows

In [ ]:
exact_duplicate_rows = int(clean_df.duplicated().sum())
exact_duplicate_rows

### 5. Check duplicate SMILES separately

Repeated SMILES may represent separate toxicity records, so SMILES alone is not used to remove rows.

In [ ]:
duplicate_smiles_before = int(clean_df['SMILES'].duplicated().sum())
duplicate_smiles_before

### 6. Remove exact duplicates

Keep the first occurrence only when every column is identical. Retain other records that share a SMILES value.

In [ ]:
rows_before_deduplication = len(clean_df)
if exact_duplicate_rows > 0:
    clean_df = clean_df.drop_duplicates()
exact_rows_removed = rows_before_deduplication - len(clean_df)
assert not clean_df.duplicated().any()
print(f'Exact duplicate rows removed: {exact_rows_removed}')
print(f'Remaining repeated SMILES values: {clean_df["SMILES"].duplicated().sum()}')

### 7. Confirm numeric values

Check that both target columns are numeric and finite. Stop for review if validation fails instead of silently changing values.

In [ ]:
import numpy as np
from pandas.api.types import is_numeric_dtype

numeric_columns = clean_df[main_columns].dtypes.apply(is_numeric_dtype)
assert numeric_columns.all(), 'LD50 and NumHAcceptors must both be numeric.'
assert np.isfinite(clean_df[main_columns]).all().all(), 'Review non-finite target values.'
clean_df[main_columns].dtypes

### 8. Validate hydrogen-bond acceptor counts

In [ ]:
acceptor_counts = clean_df['NumHAcceptors']
valid_acceptor_counts = acceptor_counts.ge(0) & acceptor_counts.mod(1).eq(0)
print(f'Invalid acceptor counts: {(~valid_acceptor_counts).sum()}')
assert valid_acceptor_counts.all(), 'Review negative or fractional NumHAcceptors values.'
clean_df['NumHAcceptors'].value_counts().sort_index()

### 9. Inspect extreme values

Inspect ranges, tail percentiles, and the five lowest and highest records for each target column. Unusual values are retained; these checks do not establish that an extreme value is an error.

In [ ]:
clean_df[main_columns].describe(percentiles=[0.01, 0.5, 0.99])

In [ ]:
from IPython.display import display

inspection_columns = ['Name', 'SMILES', 'LD50', 'NumHAcceptors']
for column in main_columns:
    print(f'Five lowest {column} values:')
    display(clean_df.nsmallest(5, column)[inspection_columns])
    print(f'Five highest {column} values:')
    display(clean_df.nlargest(5, column)[inspection_columns])

### 10. Save the cleaned dataset

Save all columns without a CSV index. Resolve the output directory from the loaded CSV path so this works from either the project root or `notebooks/`.

In [ ]:
processed_path = csv_path.parent.parent / 'processed' / 'ld50_cleaned.csv'
processed_path.parent.mkdir(parents=True, exist_ok=True)
clean_df.to_csv(processed_path, index=False)
print(f'Saved {len(clean_df):,} rows to {processed_path.as_posix()}')
print(f'Total rows removed: {initial_rows - len(clean_df)} '
      f'({missing_rows_removed} missing target values; {exact_rows_removed} exact duplicates)')

### Cleaning summary

For the current raw dataset, 7,397 rows became **7,388 rows**: **1 row** was removed for missing `LD50` (none were missing `NumHAcceptors`), followed by **8 exact duplicate rows**, for **9 rows removed in total**. There were 55 repeated SMILES values before exact deduplication; the remaining 47 are retained as potentially separate toxicity records.

Both target columns are numeric and finite, with no missing values; all `NumHAcceptors` values are non-negative whole numbers. Observed `LD50` values range from **0.291 to 7.207**, and `NumHAcceptors` from **0 to 35**. Extreme values were inspected and retained, and `LD50` was not transformed.

The final dataset is saved to `data/processed/ld50_cleaned.csv`. The original `df` and raw CSV are unchanged.